# Manual validation of the predicted epithelial cell states

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 8th January 2024\
**Last modified date:** 28th January 2024

This notebook outlines the process of validation of predicted by `scVI - scANVI` pipeline epithelial cell states annotations. We will evaluate `scANVI` classificator confidence and also validate the predicted annotation with markers.

## Import packages

In [1]:
import numpy as np
import scanpy as sc
import anndata
import pandas as pd
import plotnine as p
import matplotlib.pyplot as plt
import seaborn as sns

import json
from datetime import datetime

## Setup working environment

In [2]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi = 180, color_map = 'magma_r', dpi_save = 300, vector_friendly = True, format = 'svg')

In [3]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [4]:
timestamp

'28012025_132439'

## Upload data

In [ ]:
adata = sc.read_h5ad('integration_of_remapped_data/gut_hs_all_datasets_scVI_scANVI_epithelial_cellstates_AM_27012025_133833_raw.h5ad')
adata

/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/anndata/_core/anndata.py:1906: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.


AnnData object with n_obs × n_vars = 110105 × 43704
    obs: 'cell_id', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_protoc

## Validate confidence score and markers expression

In [6]:
adata_log = adata.copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

normalizing counts per cell. The following highly-expressed genes are not considered during normalization factor computation:
['PLA2G2A', 'GUCA2B', 'GUCA2A', 'CLCA1', 'ATP1A1-AS1', 'REG4', 'S100A6', 'ITLN1', 'ITLN2', 'APOA2', 'PIGR', 'REG1A', 'REG3A', 'FABP1', 'GCG', 'CRYBA2', 'GHRL', 'CCK', 'RBP2', 'SST', 'ENSG00000286848', 'JCHAIN', 'GC', 'ALB', 'AFP', 'RPL34', 'CCDC152', 'ATG10', 'SPINK1', 'FABP6', 'ENSG00000271581', 'MLN', 'ENSG00000287089', 'CLPS', 'AGR2', 'RPL30-AS1', 'RPS6', 'SPINK4', 'RPL12', 'LCN2', 'LCN15', 'VIM-AS1', 'ADIRF-AS1', 'SHLD2', 'INS', 'BEST1', 'FTH1', 'TALAM1', 'ENSG00000285513', 'APOA4', 'APOC3', 'APOA1', 'APOA1-AS', 'GAPDH', 'ENSG00000269968', 'MRPS35-DT', 'LYZ', 'ENSG00000257764', 'NTS', 'TPT1', 'ENSG00000273149', 'CKB', 'PHGR1', 'ENSG00000290038', 'ENSG00000290010', 'ZG16', 'MT2A', 'MT1G', 'MT1H', 'GAST', 'PYY', 'GIP', 'TTR', 'ENSG00000267598', 'YIF1B', 'RPS19', 'FTL', 'TFF3', 'TFF2', 'TFF1', 'IGLC2', 'IGLC3', 'TMSB4X', 'PCSK1N', 'MT-RNR1', 'MT-RNR2', 'MT-CO1'

In [8]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.umap(adata_log,color=["cellstates_scANVI", "confidence_score"], ncols=4, color_map = 'magma_r', frameon=False, show=False, size = 3)
    plt.savefig(f"integration_of_remapped_data/plots/manual_annotation_validation/epithelial_scANVI_confidence_{timestamp}.png", bbox_inches="tight")
    plt.show()

## Validate markers expression

In [11]:
def visualize_cell_state_markers(adata, cell_state, markers, timestamp=None):
    """
    Create temporary annotation for specific cell state and visualize with markers.
    
    Parameters:
    -----------
    adata : AnnData
        Annotated data matrix
    cell_state : str
        Cell state of interest from cellstates_scANVI column
    markers : list
        List of marker genes to visualize
    timestamp : str, optional
        Timestamp for file naming. If None, current time will be used
    """

    if timestamp is None:
        timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

    # Create temporary column for cell state
    temp_col_name = 'temp_cell_state'
    adata.obs[temp_col_name] = adata.obs['cellstates_scANVI'] == cell_state
    
    # Set up colors in uns
    original_colors = None
    if 'temp_cell_state_colors' in adata.uns:
        original_colors = adata.uns['temp_cell_state_colors'].copy()
    
    # Set up color palette
    adata.uns[f'{temp_col_name}_colors'] = ['#D3D3D3', '#ba0f30']  # Light pink for target, light grey for others
    
    # Calculate plot layout
    n_total_plots = len(markers) + 1  # +1 for cell state plot
    n_cols = 3
    n_rows = (n_total_plots + (n_cols - 1)) // n_cols  # Ceiling division
    
    # Create the plot
    with plt.rc_context():
        sc.set_figure_params(dpi=300, figsize=(15, 5*n_rows))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        
        # Convert axes to 1D array if necessary
        if n_rows == 1:
            axes = np.array([axes])  # Handle single row case
        axes = axes.flatten()
        
        # Plot cell state
        sc.pl.umap(adata, color=[temp_col_name], ax=axes[0], 
                  frameon=False, title=f'{cell_state} cells',
                  show=False, size=3)
        
        # Plot markers
        for i, marker in enumerate(markers):
            if i + 1 >= len(axes):
                break
            if marker in adata.var_names:
                sc.pl.umap(adata, color=marker, ax=axes[i + 1],
                          color_map='magma_r', frameon=False,
                          title=marker, show=False, size=3)
            else:
                print(f"Warning: Marker {marker} not found in the data")
                axes[i + 1].set_title(f"{marker} (not found)")
                axes[i + 1].axis('off')
        
        # Remove empty subplots
        for i in range(len(markers) + 1, len(axes)):
            fig.delaxes(axes[i])
        
        plt.tight_layout()
        # Save figure
        plt.savefig(
            f"integration_of_remapped_data/plots/manual_annotation_validation/{cell_state}_markers_{timestamp}.png", 
            bbox_inches="tight"
        )
        plt.show()
    
    # Clean up: remove temporary column and colors
    del adata.obs[temp_col_name]
    if f'{temp_col_name}_colors' in adata.uns:
        del adata.uns[f'{temp_col_name}_colors']
    
    # Restore original colors if they existed
    if original_colors is not None:
        adata.uns['temp_cell_state_colors'] = original_colors

+ Stem cells

In [ ]:
markers = ['LGR5','ASCL2','SMOC2','RGMB','OLFM4']
visualize_cell_state_markers(adata_log, "Stem cells", markers)

+ TA

In [ ]:
markers = ['MKI67', 'TOP2A', 'PCNA', 'CCNB1', 'CDC20', 'BIRC5', 'UBE2C']
visualize_cell_state_markers(adata_log, "TA", markers)

+ Proximal progenitor

In [ ]:
markers = ['FGG', 'BEX5', 'CLU']
visualize_cell_state_markers(adata_log, "Proximal progenitor", markers)

+ Distal progenitor

In [ ]:
markers = ['CKB', 'AKAP7', 'GPC3']
visualize_cell_state_markers(adata_log, "Distal progenitor", markers)

* Enterocyte

In [ ]:
markers = ['FABP1', 'APOA1', 'APOA4', 'RBP2','FABP2', 'CEACAM1', 'EPCAM', 'CA7', 'CA4']
visualize_cell_state_markers(adata_log, "Enterocyte", markers)

* Colonocyte

In [ ]:
markers = ['CA2', 'AQP8', 'SLC26A3', 'FABP1']
visualize_cell_state_markers(adata_log, "Colonocyte", markers)

* BEST4+ epithelial 

In [ ]:
markers = ['BEST4', 'CA7', 'OTOP2']
visualize_cell_state_markers(adata_log, "BEST4+ epithelial", markers)

* Goblet cell

In [ ]:
markers = ['MUC2', 'TFF3', 'SPINK4','CLCA1','SPDEF','FCGBP','ZG16', 'WFDC2']
visualize_cell_state_markers(adata_log, "Goblet cell", markers)

* EEC

In [23]:
adata_log.obs['cellstates_scANVI'].value_counts()

cellstates_scANVI
Enterocyte             28766
Proximal progenitor    26304
TA                     14501
Stem cells             13495
Colonocyte             11417
BEST4+ epithelial       6008
Goblet cell             5621
EEC                     3516
Distal progenitor        249
Tuft                     215
Paneth                    13
Name: count, dtype: int64

In [ ]:
markers = ['CHGA','CHGB', 'NEUROD1', 'TPH1']
visualize_cell_state_markers(adata_log, "EEC", markers)

* Paneth

In [ ]:
markers = ['LYZ', 'DEFA5', 'REG3A', 'DEFA6', 'REG4']
visualize_cell_state_markers(adata_log, "Paneth", markers)

* Tuft

In [ ]:
markers = ['POU2F3', 'TRPM5', 'IRAG2']
visualize_cell_state_markers(adata_log, "Tuft", markers)